# E1 — tuning A.1 (lr, weight decay por arquitetura)

Protocolo (seção 2.2, A.1):

- Optuna `RandomSampler`, seed 42; 8 trials por arquitetura, **os mesmos 8 pontos** para todas
  (`src/tuning.py:sample_random_trials`).
- Treino **só no fold 0** do desenvolvimento (16 pacientes); seleção pelo **Dice médio por paciente**
  na validação fixa (11 pacientes). Nenhum teste é carregado.
- 8000 passos, AdamW + Cosine (`T_max = max_steps`), batch de 4 rotulados, `binary_all`, 256×256.
- Arquiteturas: U-Net, U-Net++, FPN, DeepLabv3+ (ResNet34/ImageNet, smp) e SwinUNet tiny.

Grade: `configs/sweep/e1-tuning.yaml`. Resultado: `configs/frozen/e1-hparams.yaml`. **Fazer o commit
desse arquivo é o congelamento**: depois dele, nada de reajustar com folds avaliados ou com teste.

**Cuidado ao ler a tabela.** Re-runs idênticos do X1 divergiram até ~0,10 de Dice. Uma diferença
entre trials menor que o ruído entre re-runs não é evidência de que um ponto é melhor.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../../../")

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.sweep import Sweep, submit_sweep, sweep_status, watch_sweep
from src.tuning import rank_trials, write_frozen_hparams

SWEEP_CONFIG = "configs/sweep/e1-tuning.yaml"
FROZEN_HPARAMS = "configs/frozen/e1-hparams.yaml"

sweep = Sweep(SWEEP_CONFIG)

/home/caioseda/miniconda3/envs/vfss/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## Os pontos da busca

In [3]:
points = pd.DataFrame(sweep.plan.trial_points)
points.index.name = "trial"
points.style.format("{:.2e}")

,lr,weight_decay
trial,,
0,5.61e-05,7.11e-04
1,2.91e-04,6.25e-05
2,2.05e-05,2.94e-06
3,1.31e-05,3.97e-04
4,1.59e-04,1.33e-04
5,1.10e-05,8.12e-04
6,4.62e-04,4.34e-06
7,2.31e-05,3.55e-06


## Execução

In [4]:
sweep_status(SWEEP_CONFIG)

,run_id,seed,trial,arch,status
0,unet-t00-s42,42,0,unet,done (na CSV)
1,unetplusplus-t00-s42,42,0,unetplusplus,done (na CSV)
2,fpn-t00-s42,42,0,fpn,done (na CSV)
3,deeplabv3plus-t00-s42,42,0,deeplabv3plus,retomavel (failed)
4,swinunet_tiny-t00-s42,42,0,swinunet_tiny,pendente
5,unet-t01-s42,42,1,unet,pendente
6,unetplusplus-t01-s42,42,1,unetplusplus,pendente
7,fpn-t01-s42,42,1,fpn,pendente
8,deeplabv3plus-t01-s42,42,1,deeplabv3plus,pendente
9,swinunet_tiny-t01-s42,42,1,swinunet_tiny,pendente


In [5]:
# Lança a grade destacada (tmux) e devolve na hora. Rode SÓ depois de commitar o split.
# submit_sweep(SWEEP_CONFIG)

# Acompanha. Interromper esta célula para de OLHAR, não de treinar.
watch_sweep(SWEEP_CONFIG)

e1-tuning: 3/40 celulas concluidas (12:23:23)
               run_id  seed  trial          arch             status
         unet-t00-s42    42      0          unet      done (na CSV)
 unetplusplus-t00-s42    42      0  unetplusplus      done (na CSV)
          fpn-t00-s42    42      0           fpn      done (na CSV)
deeplabv3plus-t00-s42    42      0 deeplabv3plus retomavel (failed)
swinunet_tiny-t00-s42    42      0 swinunet_tiny           pendente
         unet-t01-s42    42      1          unet           pendente
 unetplusplus-t01-s42    42      1  unetplusplus           pendente
          fpn-t01-s42    42      1           fpn           pendente
deeplabv3plus-t01-s42    42      1 deeplabv3plus           pendente
swinunet_tiny-t01-s42    42      1 swinunet_tiny           pendente
         unet-t02-s42    42      2          unet           pendente
 unetplusplus-t02-s42    42      2  unetplusplus           pendente
          fpn-t02-s42    42      2           fpn           pendente
de

## Resultados: Dice médio por paciente na validação fixa

In [10]:
results = sweep.load_results()
assert results.empty or set(results.eval_set) == {"val"}, "o tuning so pode ter linhas de validacao"

ranked = rank_trials(results, by=["arch"])
ranked.pivot(index="trial", columns="arch", values="dice").round(4)

KeyError: 'arch'

In [ ]:
best = ranked[ranked["rank"] == 1]
best[["arch", "trial", "lr", "weight_decay", "dice"]]

## Convergência: 8000 passos bastam?

Curva de `val/dice_patient` do melhor trial de cada arquitetura. Se ela ainda estiver subindo no fim,
o protocolo pede para reavaliar `max_steps` (e `T_max` junto) **antes** de congelar.

In [ ]:
run_dirs = (results.drop_duplicates("run_id").set_index(["arch", "trial"]).output_dir)

fig, ax = plt.subplots(figsize=(8, 4))
for _, row in best.iterrows():
    metrics = pd.read_csv(Path(run_dirs[(row.arch, row.trial)]) / "csv" / "metrics.csv")
    curve = metrics.dropna(subset=["val/dice_patient"])
    ax.plot(curve.step, curve["val/dice_patient"], marker="o", label=f"{row.arch} (t{row.trial})")
ax.set_xlabel("passo"); ax.set_ylabel("val/dice_patient"); ax.legend(); ax.grid(alpha=0.3)

## Congelamento

Grava `configs/frozen/e1-hparams.yaml` (recusa sobrescrever). Commite o arquivo em seguida.

Opcional ("confirmar as duas melhores em quatro folds"): copie `configs/sweep/e1-tuning.yaml` com
outro `family`/`name`, use `data.train_folds: [1, 2, 3, 4]` e restrinja com `candidates`, que a célula
abaixo imprime.

In [ ]:
# write_frozen_hparams(best, FROZEN_HPARAMS, source=str(sweep.results_csv))

top2 = ranked[ranked["rank"] <= 2]
print("candidates:", {arch: group.trial.astype(int).tolist() for arch, group in top2.groupby("arch")})